In [1]:
from torchtext.vocab import GloVe

ModuleNotFoundError: No module named 'torchtext'

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
def learn_int8_scales(X: torch.Tensor, W: torch.Tensor, steps=200, lr=0.001, verbose=False):
    """
    X: [M,K] float16/float32
    W: [K,N] float16/float32
    Returns:
      Sx, Sw (floats), Qx_int8, Qw_int8, Yhat (float)  with Yhat ≈ X@W
    """
    dtype  = torch.float32  # optimize in fp32 for stability
    Xf = X.to(dtype)
    Wf = W.to(dtype)
    qmax = 127.0

    # sensible init from amax (symmetric)
    Sx0 = (Xf.abs().amax() / qmax).clamp(min=1e-8)
    Sw0 = (Wf.abs().amax() / qmax).clamp(min=1e-8)

    log_Sx = torch.log(Sx0.detach()).clone().requires_grad_(True)
    log_Sw = torch.log(Sw0.detach()).clone().requires_grad_(True)
    opt = torch.optim.Adam([log_Sx, log_Sw], lr=lr)

    def ste_round(x):
        # Straight-through estimator: gradient of identity, value of round
        return (x.round() - x).detach() + x

    for t in range(steps):
        Sx = torch.exp(log_Sx)
        Sw = torch.exp(log_Sw)

        # quantize with STE (float q used for matmul; integers used only at the end)
        Xn = (Xf / Sx).clamp(-qmax, qmax)
        Wn = (Wf / Sw).clamp(-qmax, qmax)
        Qx = ste_round(Xn)
        Qw = ste_round(Wn)

        # Reconstruction via product: (Qx@Qw) * (Sx*Sw)
        Y_hat = (Qx @ Qw) * (Sx * Sw)
        Y_ref = Xf @ Wf

        loss = F.mse_loss(Y_hat, Y_ref)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        if verbose and (t % max(1, steps // 10) == 0 or t == steps - 1):
            print(f"step {t:4d} | loss={loss.item():.6e} | Sx={Sx.item():.4e} Sw={Sw.item():.4e}")

    # Final integer tensors & dequant output in the exact requested form
    with torch.no_grad():
        Sx = torch.exp(log_Sx)
        Sw = torch.exp(log_Sw)
        Qx_int = (Xf / Sx).round().clamp(-qmax, qmax).to(torch.int8)
        Qw_int = (Wf / Sw).round().clamp(-qmax, qmax).to(torch.int8)

    return Sx.item(), Sw.item(), Qx_int, Qw_int


In [30]:
M, K, N = 16, 16, 16
X = torch.randn(M, K, device=device, dtype=torch.float16)
W = torch.randn(K, N, device=device, dtype=torch.float16)

ref = (X @ W)

In [31]:
Sx, Sw, Qx, Qw = learn_int8_scales(X, W, steps=1_000, lr=5e-2, verbose=True)

print(f"\nLearned Sx={Sx:.6e}, Sw={Sw:.6e}")
print(f"Qx.dtype = {Qx.dtype}, Qw.dtype = {Qw.dtype}")

step    0 | loss=1.652565e-03 | Sx=2.3807e-02 Sw=2.4160e-02
step  100 | loss=1.796238e-03 | Sx=2.3951e-02 Sw=2.4054e-02
step  200 | loss=1.614908e-03 | Sx=2.4424e-02 Sw=2.4135e-02
step  300 | loss=1.721115e-03 | Sx=2.4321e-02 Sw=2.4386e-02
step  400 | loss=1.499279e-03 | Sx=2.3972e-02 Sw=2.4499e-02
step  500 | loss=1.899250e-03 | Sx=2.3732e-02 Sw=2.4356e-02
step  600 | loss=1.536483e-03 | Sx=2.3669e-02 Sw=2.4234e-02
step  700 | loss=1.421847e-03 | Sx=2.4110e-02 Sw=2.4242e-02
step  800 | loss=1.784822e-03 | Sx=2.3814e-02 Sw=2.4045e-02
step  900 | loss=1.571380e-03 | Sx=2.3871e-02 Sw=2.4441e-02
step  999 | loss=1.857071e-03 | Sx=2.3819e-02 Sw=2.4402e-02

Learned Sx=2.397409e-02, Sw=2.451246e-02
Qx.dtype = torch.int8, Qw.dtype = torch.int8


In [33]:
Yhat = bnb.functional.int8_linear_matmul(Qx, Qw)
Yhat = Yhat.to(torch.float32) * (Sx * Sw)

err = (Yhat - ref).abs().max().item()
print(f"\nLearned Sx={Sx:.4e}, Sw={Sw:.4e}")
print("max|Yhat - XW| =", err)


Learned Sx=2.3974e-02, Sw=2.4512e-02
max|Yhat - XW| = 20.31558609008789


In [1]:
import torch

In [2]:
X = torch.randint(-128, 127, (4, 4), dtype=torch.int8)
b = torch.randint(-128, 127, (4,), dtype=torch.int8)

X + b

tensor([[-126,   51, -106,  -48],
        [  60,   26,  121,   -2],
        [ 110,  -68,  -32,  -43],
        [ -67,   61,   69,   64]], dtype=torch.int8)